<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-ComputerVision/blob/main/W04_Demo_Stitching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demo: Bildmerkmale & Image Stitching
**Computer Vision – FS 2026 | Woche 4**

Pipeline: **Detect → Match → Estimate H → Warp → Blend**


## 0  Imports

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show(images, titles, figsize=(15, 5)):
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1: axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        if img.ndim == 2:
            ax.imshow(img, cmap='gray')
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


## 1  Bilder laden

Bilder hochladen und als Liste `images` bereitstellen.
- **Lokal (Jupyter):** `ipywidgets.FileUpload` – mehrere Dateien auf einmal wählbar
- **Colab:** `google.colab.files.upload()` – erscheint automatisch

> Für den manuellen Stitching-Demo (Schritte 2–5) werden **die ersten zwei Bilder** verwendet.
> `cv2.Stitcher` in Schritt 6 verwendet **alle** hochgeladenen Bilder.

> **Tipp:** Bilder mit ~30–50 % Überlappung und ausreichend Textur liefern die besten Ergebnisse.

In [ ]:
import io

# ── Umgebung erkennen ────────────────────────────────────────────────────
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

images = []   # wird unten befüllt
filenames = []

# ── Upload ───────────────────────────────────────────────────────────────
if IN_COLAB:
    uploaded = colab_files.upload()   # öffnet Datei-Dialog im Browser
    for fname, data in uploaded.items():
        arr = np.frombuffer(data, dtype=np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        if img is not None:
            images.append(img)
            filenames.append(fname)
        else:
            print(f'⚠️  {fname} konnte nicht geladen werden – Format unterstützt?')
else:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    uploader = widgets.FileUpload(
        accept='image/*',
        multiple=True,
        description='Bilder wählen',
        layout=widgets.Layout(width='300px')
    )
    load_btn = widgets.Button(
        description='Bilder laden ✓',
        button_style='success',
        layout=widgets.Layout(width='160px')
    )
    out = widgets.Output()

    def on_load(b):
        global images, filenames
        images, filenames = [], []
        with out:
            clear_output()
            if not uploader.value:
                print('Noch keine Dateien gewählt.')
                return
            for item in uploader.value:
                fname = item['name']
                data  = item['content']
                arr   = np.frombuffer(data, dtype=np.uint8)
                img   = cv2.imdecode(arr, cv2.IMREAD_COLOR)
                if img is not None:
                    images.append(img)
                    filenames.append(fname)
                    print(f'  ✓ {fname}  ({img.shape[1]}×{img.shape[0]} px)')
                else:
                    print(f'  ✗ {fname}  – nicht lesbar')
            print(f'\n{len(images)} Bild(er) geladen.')
            if len(images) >= 2:
                show(images[:4],
                     [f'{fn}' for fn in filenames[:4]],
                     figsize=(min(5 * len(images[:4]), 20), 4))
            if len(images) < 2:
                print('⚠️  Bitte mindestens 2 Bilder hochladen.')

    load_btn.on_click(on_load)
    display(widgets.HBox([uploader, load_btn]), out)


## 2  Schritt 1: DETECT – Keypoints & Descriptors
SIFT (Scale-Invariant Feature Transform) erkennt markante Punkte und beschreibt deren lokale Umgebung als 128-dimensionalen Vektor.


In [ ]:
if len(images) < 2:
    print('⚠️  Bitte zuerst mindestens 2 Bilder hochladen (Zelle oben).')
else:
    img1, img2 = images[0], images[1]

    # SIFT initialisieren
    sift = cv2.SIFT_create(nfeatures=500)

    # Keypoints und Deskriptoren berechnen
    kp1, des1 = sift.detectAndCompute(img1, None)
    kp2, des2 = sift.detectAndCompute(img2, None)

    # Keypoints einzeichnen
    img1_kp = cv2.drawKeypoints(img1, kp1, None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    img2_kp = cv2.drawKeypoints(img2, kp2, None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

    show([img1_kp, img2_kp],
         [f'{filenames[0]}: {len(kp1)} Keypoints',
          f'{filenames[1]}: {len(kp2)} Keypoints'])
    print(f'Deskriptor-Dimension: {des1.shape[1]}D (SIFT)')


## 3  Schritt 2: MATCH – Feature Matching + Ratio-Test
Der **Lowe's Ratio Test** filtert mehrdeutige Matches:  
Ein Match gilt als gut, wenn der beste Nachbar deutlich besser als der zweitbeste ist.


In [ ]:
if len(images) >= 2 and 'des1' in dir():
    # BFMatcher (Brute-Force) mit k=2 Nachbarn
    bf = cv2.BFMatcher(cv2.NORM_L2)
    matches_raw = bf.knnMatch(des1, des2, k=2)

    # Ratio-Test (Lowe 2004)
    ratio_threshold = 0.75
    good_matches = [m for m, n in matches_raw if m.distance < ratio_threshold * n.distance]

    print(f'Rohe Matches:    {len(matches_raw)}')
    print(f'Nach Ratio-Test: {len(good_matches)}  (threshold={ratio_threshold})')

    # Matches visualisieren
    img_matches = cv2.drawMatches(img1, kp1, img2, kp2,
        good_matches[:40], None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

    plt.figure(figsize=(16, 5))
    plt.imshow(cv2.cvtColor(img_matches, cv2.COLOR_BGR2RGB))
    plt.title(f'Feature Matches nach Ratio-Test (zeige erste 40 von {len(good_matches)})')
    plt.axis('off')
    plt.show()


## 4  Schritt 3: ESTIMATE H – Homographie mit RANSAC
**RANSAC** (Random Sample Consensus) schätzt die Homographie robust,  
indem es Ausreisser (falsche Matches) automatisch ignoriert.


In [ ]:
if len(images) >= 2 and 'good_matches' in dir():
    MIN_MATCH_COUNT = 10

    if len(good_matches) >= MIN_MATCH_COUNT:
        src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1,1,2)
        dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1,1,2)

        H, mask = cv2.findHomography(dst_pts, src_pts,
                                       cv2.RANSAC, ransacReprojThreshold=5.0)
        inliers  = int(mask.sum())
        outliers = len(mask) - inliers

        print(f'Homographie erfolgreich geschätzt')
        print(f'Inlier:  {inliers} ({100*inliers/len(mask):.0f}%)')
        print(f'Outlier: {outliers}')
        print(f'\nHomographie-Matrix H:')
        print(np.round(H, 4))
    else:
        print(f'Zu wenige Matches: {len(good_matches)} < {MIN_MATCH_COUNT}')
        H = None


## 6  Alternative: cv2.Stitcher (High-Level API)


In [ ]:
if len(images) < 2:
    print('⚠️  Bitte zuerst Bilder hochladen.')
else:
    print(f'cv2.Stitcher verwendet alle {len(images)} Bilder ...')
    stitcher = cv2.Stitcher_create(cv2.Stitcher_PANORAMA)
    status, panorama = stitcher.stitch(images)

    status_msg = {
        cv2.Stitcher_OK:                       '✅ Erfolgreich',
        cv2.Stitcher_ERR_NEED_MORE_IMGS:        '❌ Zu wenige Bilder',
        cv2.Stitcher_ERR_HOMOGRAPHY_EST_FAIL:   '❌ Homographie fehlgeschlagen',
        cv2.Stitcher_ERR_CAMERA_PARAMS_ADJUST_FAIL: '❌ Kameraparameter-Fehler',
    }
    print(f'Status: {status_msg.get(status, str(status))}')

    if status == cv2.Stitcher_OK:
        show([panorama], [f'cv2.Stitcher Panorama ({len(images)} Bilder)'],
             figsize=(18, 6))
        print(f'Panorama-Grösse: {panorama.shape[1]}×{panorama.shape[0]} px')
    else:
        print('Tipps:')
        print('  - Ausreichend Überlappung (30–50 %)?')
        print('  - Bilder in der richtigen Reihenfolge (links → rechts)?')
        print('  - Genug Textur/Kontrast im Überlappungsbereich?')


---
## Zusammenfassung der Pipeline
| Schritt | Funktion | Wichtige Parameter |
|---|---|---|
| Detect | `SIFT_create(nfeatures=...)` | Anzahl Features |
| Match | `BFMatcher` + Ratio-Test | ratio_threshold (0.7–0.8) |
| Estimate H | `findHomography(..., cv2.RANSAC)` | ransacReprojThreshold |
| Warp | `warpPerspective` | Ausgabegrösse |
| Blend | Einfaches Überlappen oder Alpha-Blending | — |
